# Final EDA, Column Cleaning, Missing-Value Plan, and Leakage-Safe Split

This notebook prepares the final thesis dataset before modelling.

Main goals:
1. Load the final clean expanded PX4 dataset.
2. Remove columns that are 100% empty.
3. Create a column report explaining missing percentage, correlation with target, and feature decision.
4. Finalise EDA with only useful plots/tables.
5. Define how missing values will be handled.
6. Create a final modelling dataset.
7. Split data by `log_id` to avoid time-series leakage.
8. Impute missing values only after splitting, using training data statistics only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

# Main project path
PROJECT_DIR = Path("/Users/arjun/Documents/UAV thesis/uav_px4_dataset_project")

# Input dataset
DATA_PATH = PROJECT_DIR / "processed_10hz" / "clean_ml_ready_10hz_expanded.csv"

# Output folders
EDA_DIR = PROJECT_DIR / "eda_outputs"
PROCESSED_DIR = PROJECT_DIR / "processed_10hz"
SPLIT_DIR = PROJECT_DIR / "splits"

EDA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(f"File not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Loaded successfully")
print("Shape:", df.shape)
print("Unique flights:", df["log_id"].nunique())
print("\nDataset label distribution:")
print(df["dataset_label"].value_counts())
display(df.head())

## 1. Final Dataset Summary

This table documents the final dataset after `.ulg` extraction, 10 Hz resampling, zero-current/zero-power flight cleaning, and creation of the 60-second energy target.

In [ ]:
summary_table = pd.DataFrame({
    "Metric": [
        "Final clean rows",
        "Number of columns",
        "Unique flight logs",
        "Normal rows",
        "Abnormal rows",
        "Normal flight logs",
        "Abnormal flight logs",
        "Sampling rate",
        "Main target"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        df["log_id"].nunique(),
        df[df["dataset_label"] == "normal"].shape[0],
        df[df["dataset_label"] == "abnormal"].shape[0],
        df[df["dataset_label"] == "normal"]["log_id"].nunique(),
        df[df["dataset_label"] == "abnormal"]["log_id"].nunique(),
        "10 Hz",
        "target_energy_next_60s_wh"
    ]
})

display(summary_table)
summary_table.to_csv(EDA_DIR / "01_final_dataset_summary.csv", index=False)

## 2. Core Column Meaning and Missingness

These are the columns that must be protected because they define the flight, time, battery-energy behaviour, and prediction target.

In [ ]:
core_columns_dictionary = pd.DataFrame([
    {"column": "log_id", "meaning": "Unique identifier for each PX4 flight log", "use": "Used for flight-level splitting; not used as model input"},
    {"column": "dataset_label", "meaning": "Normal or abnormal label based on PX4 rating group", "use": "Used for EDA and stratified splitting; not used as model input unless explicitly testing label-based models"},
    {"column": "time_td", "meaning": "Time delta from start of flight in timedelta format", "use": "Reference only; not used as model input"},
    {"column": "time_s", "meaning": "Time in seconds from the start of each flight", "use": "Candidate feature; useful for flight progress"},
    {"column": "battery_status__voltage_v", "meaning": "Raw battery voltage in volts", "use": "Core battery feature"},
    {"column": "battery_status__voltage_filtered_v", "meaning": "Filtered/smoothed battery voltage", "use": "Core battery feature"},
    {"column": "battery_status__current_a", "meaning": "Battery current draw in amperes", "use": "Core battery feature"},
    {"column": "power_w", "meaning": "Instantaneous power calculated as voltage × absolute current", "use": "Core energy feature"},
    {"column": "energy_increment_wh", "meaning": "Energy consumed during each 0.1 second step at 10 Hz", "use": "Derived energy feature; usually not used with target if too direct"},
    {"column": "cumulative_energy_wh", "meaning": "Total energy consumed from start of flight to current row", "use": "Important flight-progress energy feature"},
    {"column": "future_cumulative_energy_wh", "meaning": "Cumulative energy 60 seconds in the future", "use": "Leakage column; used only to create target, not model input"},
    {"column": "target_energy_next_60s_wh", "meaning": "Energy consumed in the next 60 seconds", "use": "Main prediction target"}
])

# Add missingness information
core_columns_dictionary["exists"] = core_columns_dictionary["column"].apply(lambda c: c in df.columns)
core_columns_dictionary["missing_count"] = core_columns_dictionary["column"].apply(
    lambda c: df[c].isna().sum() if c in df.columns else np.nan
)
core_columns_dictionary["missing_percent"] = core_columns_dictionary["column"].apply(
    lambda c: df[c].isna().mean() * 100 if c in df.columns else np.nan
)

display(core_columns_dictionary)
core_columns_dictionary.to_csv(EDA_DIR / "02_core_column_dictionary.csv", index=False)

## 3. All-Column Missing Value Report

This creates a full column-level report. It is useful for explaining which columns were removed and why.

In [ ]:
missing_summary = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().sum().values / len(df)) * 100,
    "non_missing_count": df.notna().sum().values,
    "non_missing_percent": (df.notna().sum().values / len(df)) * 100,
    "unique_values": df.nunique(dropna=True).values
}).sort_values("missing_percent", ascending=False).reset_index(drop=True)

missing_summary["missing_percent"] = missing_summary["missing_percent"].round(3)
missing_summary["non_missing_percent"] = missing_summary["non_missing_percent"].round(3)

display(missing_summary.head(50))

# Save as CSV and Excel
missing_summary.to_csv(EDA_DIR / "03_all_columns_missing_report.csv", index=False)

with pd.ExcelWriter(EDA_DIR / "03_all_columns_missing_report.xlsx", engine="openpyxl") as writer:
    missing_summary.to_excel(writer, sheet_name="All_Columns", index=False)
    missing_summary[missing_summary["missing_percent"] == 100].to_excel(writer, sheet_name="Fully_Missing_100", index=False)
    missing_summary[missing_summary["missing_percent"] >= 80].to_excel(writer, sheet_name="High_Missing_80plus", index=False)
    missing_summary[missing_summary["missing_percent"] < 20].to_excel(writer, sheet_name="Low_Missing_Under20", index=False)

print("Missing report saved to:", EDA_DIR / "03_all_columns_missing_report.xlsx")

## 4. Remove 100% Empty Columns

Reason: columns with 100% missing values contain no information and cannot help modelling or EDA.
This creates a cleaner dataset while keeping all rows.

In [ ]:
fully_missing_cols = missing_summary.loc[
    missing_summary["missing_percent"] == 100, "column"
].tolist()

df_no_empty = df.drop(columns=fully_missing_cols)

NO_EMPTY_OUTPUT = PROCESSED_DIR / "clean_ml_ready_10hz_expanded_no_empty_cols.csv"
df_no_empty.to_csv(NO_EMPTY_OUTPUT, index=False)

print("100% empty columns removed:", len(fully_missing_cols))
print("Original shape:", df.shape)
print("After removing 100% empty columns:", df_no_empty.shape)
print("Saved to:", NO_EMPTY_OUTPUT)

pd.DataFrame({"dropped_100_percent_empty_columns": fully_missing_cols}).to_csv(
    EDA_DIR / "04_dropped_100_percent_empty_columns.csv",
    index=False
)

## 5. Column Role, Correlation, and Feature Decision Report

This is the main method used to decide what each column does and whether it should be kept.

Decision method:
1. Protect core identifier and target columns.
2. Remove leakage columns such as future cumulative energy.
3. Remove columns with 100% missing values.
4. Mark columns with high missing values for removal from modelling.
5. Use correlation with the target as a ranking aid, not as the only selection rule.
6. Keep domain-relevant battery, energy, motion, position, and attitude features when data quality is acceptable.

In [ ]:
TARGET_COL = "target_energy_next_60s_wh"

ID_COLS = ["log_id"]
LABEL_COLS = ["dataset_label"]
TIME_REFERENCE_COLS = ["time_td"]
CORE_FEATURE_COLS = [
    "time_s",
    "battery_status__voltage_v",
    "battery_status__voltage_filtered_v",
    "battery_status__current_a",
    "battery_status__current_filtered_a",
    "battery_status__current_average_a",
    "battery_status__remaining",
    "battery_status__discharged_mah",
    "power_w",
    "cumulative_energy_wh"
]
DERIVED_BUT_RISKY_COLS = ["energy_increment_wh"]  # very close to power; useful for EDA, optional for modelling
LEAKAGE_COLS = [
    "future_cumulative_energy_wh",
    "future_remaining_60s",
    "target_remaining_drop_60s",
    TARGET_COL
]

def infer_column_group(col):
    c = col.lower()
    if col in ID_COLS:
        return "identifier"
    if col in LABEL_COLS:
        return "label"
    if col in TIME_REFERENCE_COLS:
        return "time_reference"
    if col == TARGET_COL:
        return "target"
    if col in LEAKAGE_COLS:
        return "leakage_or_target"
    if "battery_status" in c:
        return "battery"
    if "power" in c or "energy" in c:
        return "energy"
    if "vehicle_local_position" in c or "vehicle_gps_position" in c:
        return "position_gps"
    if "attitude" in c:
        return "attitude"
    if "acceleration" in c or "angular_velocity" in c or "sensor_combined" in c:
        return "motion_imu"
    if "actuator" in c or "motor" in c:
        return "actuator_motor"
    if "vehicle_status" in c:
        return "vehicle_status"
    if "air_data" in c or "airspeed" in c:
        return "air_data"
    return "other"

# Correlation with target for numeric columns only
numeric_cols = df_no_empty.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = {}
if TARGET_COL in numeric_cols:
    corr_series = df_no_empty[numeric_cols].corr(numeric_only=True)[TARGET_COL]
    corr_with_target = corr_series.to_dict()

column_decisions = missing_summary.copy()
column_decisions["column_group"] = column_decisions["column"].apply(infer_column_group)
column_decisions["correlation_with_target"] = column_decisions["column"].map(corr_with_target)
column_decisions["abs_correlation_with_target"] = column_decisions["correlation_with_target"].abs()

HIGH_MISSING_THRESHOLD = 80

def decide_column(row):
    col = row["column"]
    miss = row["missing_percent"]
    group = row["column_group"]

    if col in ID_COLS:
        return "keep_for_split_not_model"
    if col in LABEL_COLS:
        return "keep_for_eda_stratification_not_model"
    if col in TIME_REFERENCE_COLS:
        return "drop_from_model_keep_reference"
    if col == TARGET_COL:
        return "target"
    if col in LEAKAGE_COLS:
        return "drop_leakage"
    if miss == 100:
        return "drop_100_percent_missing"
    if miss >= HIGH_MISSING_THRESHOLD:
        return "drop_high_missing"
    if row["unique_values"] <= 1:
        return "drop_constant"
    if col in CORE_FEATURE_COLS:
        return "keep_core_feature"
    if col in DERIVED_BUT_RISKY_COLS:
        return "optional_derived_feature_review"
    if group in ["battery", "energy", "position_gps", "attitude", "motion_imu", "actuator_motor", "air_data"]:
        return "candidate_feature"
    return "review_or_drop_low_priority"

column_decisions["decision"] = column_decisions.apply(decide_column, axis=1)

display(column_decisions.sort_values(["decision", "missing_percent"]).head(80))

column_decisions.to_csv(EDA_DIR / "05_column_decision_report.csv", index=False)
with pd.ExcelWriter(EDA_DIR / "05_column_decision_report.xlsx", engine="openpyxl") as writer:
    column_decisions.to_excel(writer, sheet_name="All_Decisions", index=False)
    column_decisions[column_decisions["decision"].str.contains("drop", na=False)].to_excel(writer, sheet_name="Dropped_or_Excluded", index=False)
    column_decisions[column_decisions["decision"].str.contains("keep|candidate|optional", na=False)].to_excel(writer, sheet_name="Kept_or_Candidate", index=False)

print("Column decision report saved to:", EDA_DIR / "05_column_decision_report.xlsx")

## 6. Final EDA: Flight-Level Summary

This checks whether the dataset has a meaningful normal/abnormal split and whether the cleaned flights contain real energy behaviour.

In [ ]:
current_col = "battery_status__current_a"

flight_summary = (
    df_no_empty.groupby(["log_id", "dataset_label"])
    .agg(
        rows=("time_s", "count"),
        duration_s=("time_s", "max"),
        mean_voltage_v=("battery_status__voltage_v", "mean"),
        mean_current_a=(current_col, "mean"),
        max_current_a=(current_col, "max"),
        mean_power_w=("power_w", "mean"),
        max_power_w=("power_w", "max"),
        total_energy_wh=("cumulative_energy_wh", "max"),
        mean_target_60s=(TARGET_COL, "mean")
    )
    .reset_index()
)

flight_summary["duration_min"] = flight_summary["duration_s"] / 60

# Zero-current/zero-power verification after cleaning
zero_stats = (
    df_no_empty.assign(
        zero_current=df_no_empty[current_col].abs() < 0.01,
        zero_power=df_no_empty["power_w"].abs() < 0.01
    )
    .groupby(["log_id", "dataset_label"])
    .agg(
        zero_current_percent=("zero_current", "mean"),
        zero_power_percent=("zero_power", "mean")
    )
    .reset_index()
)

zero_stats["zero_current_percent"] *= 100
zero_stats["zero_power_percent"] *= 100

flight_summary = flight_summary.merge(zero_stats, on=["log_id", "dataset_label"], how="left")

display(flight_summary.head())

flight_level_by_label = (
    flight_summary.groupby("dataset_label")
    .agg(
        flights=("log_id", "count"),
        rows=("rows", "sum"),
        mean_duration_min=("duration_min", "mean"),
        median_duration_min=("duration_min", "median"),
        mean_power_w=("mean_power_w", "mean"),
        mean_total_energy_wh=("total_energy_wh", "mean"),
        mean_target_60s=("mean_target_60s", "mean"),
        mean_zero_current_percent=("zero_current_percent", "mean")
    )
    .reset_index()
)

display(flight_level_by_label)

flight_summary.to_csv(EDA_DIR / "06_flight_level_summary.csv", index=False)
flight_level_by_label.to_csv(EDA_DIR / "06_flight_level_summary_by_label.csv", index=False)

## 7. Final EDA Plot: Target Distribution

This plot directly supports the research question because it shows how next-60-second energy consumption differs between normal and abnormal flights.

In [ ]:
plt.figure(figsize=(10, 5))
for label in ["normal", "abnormal"]:
    temp = df_no_empty[df_no_empty["dataset_label"] == label][TARGET_COL].dropna()
    plt.hist(temp, bins=80, alpha=0.55, label=label)

plt.xlabel("Energy consumption in next 60 seconds (Wh)")
plt.ylabel("Frequency")
plt.title("Target distribution by flight label")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(EDA_DIR / "07_target_distribution_by_label.png", dpi=200)
plt.show()

target_summary_by_label = (
    df_no_empty.groupby("dataset_label")[TARGET_COL]
    .describe()
    .reset_index()
)

display(target_summary_by_label)
target_summary_by_label.to_csv(EDA_DIR / "07_target_summary_by_label.csv", index=False)

## 8. Final EDA Plot: Battery/Energy Behaviour for Example Flights

Only one normal and one abnormal flight are plotted to avoid unnecessary plots. This tells a clear story about how voltage, current, power, and cumulative energy change over time.

In [ ]:
normal_flight = df_no_empty[df_no_empty["dataset_label"] == "normal"]["log_id"].iloc[0]
abnormal_flight = df_no_empty[df_no_empty["dataset_label"] == "abnormal"]["log_id"].iloc[0]

for flight_id in [normal_flight, abnormal_flight]:
    temp = df_no_empty[df_no_empty["log_id"] == flight_id].sort_values("time_s")
    label = temp["dataset_label"].iloc[0]

    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    axes[0].plot(temp["time_s"], temp["battery_status__voltage_v"])
    axes[0].set_ylabel("Voltage (V)")
    axes[0].set_title(f"Battery and energy behaviour over time ({label} flight)")

    axes[1].plot(temp["time_s"], temp["battery_status__current_a"])
    axes[1].set_ylabel("Current (A)")

    axes[2].plot(temp["time_s"], temp["power_w"])
    axes[2].set_ylabel("Power (W)")

    axes[3].plot(temp["time_s"], temp["cumulative_energy_wh"])
    axes[3].set_ylabel("Cumulative Energy (Wh)")
    axes[3].set_xlabel("Time (seconds)")

    for ax in axes:
        ax.grid(True)

    plt.tight_layout()
    plt.savefig(EDA_DIR / f"08_battery_energy_behaviour_{label}_{flight_id}.png", dpi=200)
    plt.show()

## 9. Feature Relevance Ranking

This is not the final model feature importance. It is an EDA-level ranking using correlation with the target. It helps identify which variables are most related to next-60-second energy consumption.

Important: correlation is only a first check. Random Forest/XGBoost feature importance and permutation importance should be used later after model training.

In [ ]:
# Exclude target and leakage columns from relevance ranking
corr_feature_report = column_decisions[
    ~column_decisions["decision"].isin(["target", "drop_leakage", "drop_100_percent_missing"])
].copy()

corr_feature_report = corr_feature_report[
    corr_feature_report["correlation_with_target"].notna()
].sort_values("abs_correlation_with_target", ascending=False)

display(corr_feature_report[[
    "column", "column_group", "missing_percent", "unique_values",
    "correlation_with_target", "decision"
]].head(40))

corr_feature_report.to_csv(EDA_DIR / "09_feature_correlation_ranking.csv", index=False)

plt.figure(figsize=(10, 6))
top_corr = corr_feature_report.head(20).iloc[::-1]
plt.barh(top_corr["column"], top_corr["abs_correlation_with_target"])
plt.xlabel("Absolute correlation with target")
plt.title("Top EDA-level feature correlations with target")
plt.tight_layout()
plt.savefig(EDA_DIR / "09_top_feature_correlations.png", dpi=200)
plt.show()

## 10. Create Final Model-Ready Dataset Before Splitting

This dataset:
- keeps identifiers needed for splitting,
- keeps the target,
- removes 100% empty columns,
- removes high-missing columns,
- removes leakage columns from feature set,
- keeps only columns that can be justified for modelling.

Missing values are **not imputed here** because imputation must be fitted only on the training set after splitting.

In [ ]:
# Columns allowed in the pre-split modelling dataset
keep_decisions = [
    "keep_for_split_not_model",
    "keep_for_eda_stratification_not_model",
    "target",
    "keep_core_feature",
    "candidate_feature",
    "optional_derived_feature_review"
]

model_base_cols = column_decisions[
    column_decisions["decision"].isin(keep_decisions)
]["column"].tolist()

# Remove columns that no longer exist after 100% missing drop
model_base_cols = [c for c in model_base_cols if c in df_no_empty.columns]

# Explicitly remove leakage columns except target
model_base_cols = [
    c for c in model_base_cols
    if c not in ["future_cumulative_energy_wh", "future_remaining_60s", "target_remaining_drop_60s"]
]

# Ensure core columns are included
required_cols = ["log_id", "dataset_label", "time_s", TARGET_COL]
for c in required_cols:
    if c in df_no_empty.columns and c not in model_base_cols:
        model_base_cols.append(c)

model_base_df = df_no_empty[model_base_cols].copy()

MODEL_BASE_OUTPUT = PROCESSED_DIR / "final_model_ready_before_split.csv"
model_base_df.to_csv(MODEL_BASE_OUTPUT, index=False)

print("Final model-ready dataset before split saved to:", MODEL_BASE_OUTPUT)
print("Shape:", model_base_df.shape)
print("Unique flights:", model_base_df["log_id"].nunique())
print(model_base_df["dataset_label"].value_counts())

pd.DataFrame({"model_base_columns": model_base_cols}).to_csv(
    EDA_DIR / "10_final_model_base_columns.csv",
    index=False
)

## 11. Missing-Value Preprocessing Plan

The plan:
1. Drop columns with 100% missing values.
2. Drop high-missing columns (≥80%) from modelling.
3. Keep core battery/target columns because they have 0% missing.
4. Do not impute before splitting.
5. After splitting by `log_id`, fit median imputation only on training data.
6. Apply the trained imputer to validation and test sets.
This avoids data leakage.

In [ ]:
model_missing_summary = pd.DataFrame({
    "column": model_base_df.columns,
    "data_type": model_base_df.dtypes.astype(str).values,
    "missing_count": model_base_df.isna().sum().values,
    "missing_percent": (model_base_df.isna().sum().values / len(model_base_df)) * 100,
    "unique_values": model_base_df.nunique(dropna=True).values
}).sort_values("missing_percent", ascending=False).reset_index(drop=True)

display(model_missing_summary.head(50))
model_missing_summary.to_csv(EDA_DIR / "11_model_base_missing_summary.csv", index=False)

## 12. Leakage-Safe Train/Validation/Test Split by `log_id`

Reason:
Rows from the same flight are highly similar. If random row splitting is used, the model may see part of a flight during training and another part during testing. That would create data leakage.

Solution:
Split by `log_id`, so each complete flight appears in only one set.

In [ ]:
from sklearn.model_selection import train_test_split

unique_flights = model_base_df[["log_id", "dataset_label"]].drop_duplicates()

print("Unique flights by label:")
display(unique_flights["dataset_label"].value_counts())

train_flights, temp_flights = train_test_split(
    unique_flights,
    test_size=0.30,
    random_state=42,
    stratify=unique_flights["dataset_label"]
)

val_flights, test_flights = train_test_split(
    temp_flights,
    test_size=0.50,
    random_state=42,
    stratify=temp_flights["dataset_label"]
)

train_df = model_base_df[model_base_df["log_id"].isin(train_flights["log_id"])].copy()
val_df = model_base_df[model_base_df["log_id"].isin(val_flights["log_id"])].copy()
test_df = model_base_df[model_base_df["log_id"].isin(test_flights["log_id"])].copy()

print("Train rows:", train_df.shape)
print("Validation rows:", val_df.shape)
print("Test rows:", test_df.shape)

print("\nTrain flights:", train_df["log_id"].nunique())
print("Validation flights:", val_df["log_id"].nunique())
print("Test flights:", test_df["log_id"].nunique())

display(pd.DataFrame({
    "train": train_flights["dataset_label"].value_counts(),
    "validation": val_flights["dataset_label"].value_counts(),
    "test": test_flights["dataset_label"].value_counts()
}))

## 13. Leakage Check

All overlap values must be 0.

In [ ]:
train_ids = set(train_df["log_id"])
val_ids = set(val_df["log_id"])
test_ids = set(test_df["log_id"])

print("Train-Val overlap:", len(train_ids.intersection(val_ids)))
print("Train-Test overlap:", len(train_ids.intersection(test_ids)))
print("Val-Test overlap:", len(val_ids.intersection(test_ids)))

## 14. Train-Only Imputation for Modelling

This creates imputed modelling files. The imputer is fitted only on the training set and then applied to validation and test sets.

Note:
`log_id`, `dataset_label`, and target are kept separately. Numeric features are imputed using the training median.

In [ ]:
from sklearn.impute import SimpleImputer

target_col = TARGET_COL
non_feature_cols = ["log_id", "dataset_label", target_col]

# Numeric features only
feature_cols = [
    c for c in train_df.select_dtypes(include=[np.number]).columns
    if c not in non_feature_cols
]

# Remove any remaining leakage columns
feature_cols = [
    c for c in feature_cols
    if c not in ["future_cumulative_energy_wh", "future_remaining_60s", "target_remaining_drop_60s"]
]

print("Number of modelling features:", len(feature_cols))
display(pd.DataFrame({"feature_col": feature_cols}).head(50))

imputer = SimpleImputer(strategy="median")

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=feature_cols,
    index=train_df.index
)

X_val_imputed = pd.DataFrame(
    imputer.transform(X_val),
    columns=feature_cols,
    index=val_df.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=feature_cols,
    index=test_df.index
)

train_imputed = pd.concat([train_df[non_feature_cols].reset_index(drop=True), X_train_imputed.reset_index(drop=True)], axis=1)
val_imputed = pd.concat([val_df[non_feature_cols].reset_index(drop=True), X_val_imputed.reset_index(drop=True)], axis=1)
test_imputed = pd.concat([test_df[non_feature_cols].reset_index(drop=True), X_test_imputed.reset_index(drop=True)], axis=1)

print("Train imputed shape:", train_imputed.shape)
print("Validation imputed shape:", val_imputed.shape)
print("Test imputed shape:", test_imputed.shape)

print("Any missing in train features:", train_imputed[feature_cols].isna().sum().sum())
print("Any missing in val features:", val_imputed[feature_cols].isna().sum().sum())
print("Any missing in test features:", test_imputed[feature_cols].isna().sum().sum())

## 15. Save Final Pre-Modelling Files

These are the final outputs before model training.

In [ ]:
# Save raw split before imputation
train_df.to_csv(SPLIT_DIR / "train_model_base_before_imputation.csv", index=False)
val_df.to_csv(SPLIT_DIR / "val_model_base_before_imputation.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test_model_base_before_imputation.csv", index=False)

# Save imputed split for baseline modelling
train_imputed.to_csv(SPLIT_DIR / "train_model_ready_imputed.csv", index=False)
val_imputed.to_csv(SPLIT_DIR / "val_model_ready_imputed.csv", index=False)
test_imputed.to_csv(SPLIT_DIR / "test_model_ready_imputed.csv", index=False)

# Save flight IDs
train_flights.to_csv(SPLIT_DIR / "train_log_ids.csv", index=False)
val_flights.to_csv(SPLIT_DIR / "val_log_ids.csv", index=False)
test_flights.to_csv(SPLIT_DIR / "test_log_ids.csv", index=False)

# Save feature list
pd.DataFrame({"feature_col": feature_cols}).to_csv(SPLIT_DIR / "model_feature_columns.csv", index=False)

print("Saved final pre-modelling files to:", SPLIT_DIR)

## 16. Final Method Summary for Thesis

Use this section as the explanation of what was done and why.

In [ ]:
method_summary = pd.DataFrame([
    {
        "stage": "Remove 100% missing columns",
        "decision": "Dropped columns that contained no values in the final dataset.",
        "reason": "These columns contain no usable information and cannot support EDA or modelling."
    },
    {
        "stage": "High missing-value handling",
        "decision": "Columns with 80% or more missing values were excluded from modelling.",
        "reason": "Very sparse columns are unreliable and would require excessive imputation."
    },
    {
        "stage": "Core feature protection",
        "decision": "Battery voltage, current, power, cumulative energy, time, and target columns were protected.",
        "reason": "These variables directly describe UAV battery-energy behaviour and had 0% missing values."
    },
    {
        "stage": "Feature relevance method",
        "decision": "Used domain relevance, missingness, leakage checks, variance, and correlation ranking.",
        "reason": "This gives a defensible feature-selection method before model-based feature importance."
    },
    {
        "stage": "Imputation",
        "decision": "Median imputation fitted only on training data after log-level splitting.",
        "reason": "This avoids using validation/test information during preprocessing."
    },
    {
        "stage": "Data splitting",
        "decision": "Split by log_id instead of random rows.",
        "reason": "Rows inside the same flight are time-correlated; splitting by flight avoids data leakage."
    }
])

display(method_summary)
method_summary.to_csv(EDA_DIR / "16_method_summary_for_thesis.csv", index=False)